In [16]:
import re
import numpy as np
import pandas as pd

In [17]:
with open("..\Corpus\corpus_RNN_Médine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

### I will need to regroup each song and then count the number of occurences from each of my parts

Ok some of the songs seems empty in parts but can contain's some text 

In [18]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus)]

In [19]:
for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                if all_parts[nn][1] - all_parts[nn-2][1] > 1000 :
                    print(all_parts[nn-2],all_parts[nn])

('BEGINNING', 571533, 571544) ('END', 573523, 573528)
('BEGINNING', 565661, 565672) ('END', 569979, 569984)
('BEGINNING', 562181, 562192) ('END', 564142, 564147)
('BEGINNING', 521260, 521271) ('END', 525686, 525691)
('BEGINNING', 513709, 513720) ('END', 515270, 515275)
('BEGINNING', 496319, 496330) ('END', 500939, 500944)
('BEGINNING', 305544, 305555) ('END', 307365, 307370)
('BEGINNING', 302677, 302688) ('END', 305537, 305542)
('BEGINNING', 218798, 218809) ('END', 222450, 222455)
('BEGINNING', 198469, 198480) ('END', 200510, 200515)
('BEGINNING', 177338, 177349) ('END', 183239, 183244)
('BEGINNING', 82198, 82209) ('END', 86917, 86922)


In [20]:
for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                if all_parts[nn][1] - all_parts[nn-2][1] < 1000 :
                    print(all_parts[nn-2],all_parts[nn])
                    part_1_corp = corpus[:all_parts[nn-2][1]]
                    part_2_corp = corpus[all_parts[nn][2]:]
                    corpus = part_1_corp+part_2_corp

('BEGINNING', 584290, 584301) ('END', 585044, 585049)
('BEGINNING', 574673, 574684) ('END', 575448, 575453)
('BEGINNING', 564149, 564160) ('END', 564764, 564769)
('BEGINNING', 465972, 465983) ('END', 466621, 466626)


In [21]:
corpus_markov = corpus

In [22]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_markov)]

count = 0
beg = 0
endi =0

for i in all_parts :
    if i[0] == "BEGINNING" :
        count = 0
        beg = i[1]
    else : 
        if i[0] == "END" :
            count+=1
            endi = i[2]
            if count >=100 :
                print(count, beg, endi)
                break
        else : 
            count+=1

corpus_markov = corpus_markov[:beg] + corpus_markov[endi:]

321 439486 443946


A song has 321 parts in it, after inspection it's not an error from my code : http://genius.com/Artistes-divers-morts-pour-rien-lyrics

It's an outlier that can be explained by the fact that it's related to a specific event that happened in 2005

### Featuring will be removed from the Markov Chain as this model will only generate the lyrics of one artist, so the structure of the songs can't be of a featuring.

In [23]:
print("Nb of songs :",len(re.findall(r"<END>", corpus_markov)), "Nb of featurings : ",len(re.findall(r"<BEGINNING>True.*?<END>", corpus_markov, flags=re.DOTALL)))

Nb of songs : 240 Nb of featurings :  96


In [24]:
corpus_solo = re.sub(r"<BEGINNING>True.*?<END>\n\n", " ", corpus_markov, flags=re.DOTALL)

In [25]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

### We need to delete the songs where only one part has been identified

Most of the time that can be explained by an error in collecting the parts because the format can changed during the year of release

In [26]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                print(all_parts[nn-2], all_parts[nn])
                part_1_corp = corpus_solo[:all_parts[nn-2][1]]
                part_2_corp = corpus_solo[all_parts[nn][2]:]
                corpus_solo = part_1_corp+part_2_corp

('BEGINNING', 430268, 430279) ('END', 432258, 432263)
('BEGINNING', 424396, 424407) ('END', 428714, 428719)
('BEGINNING', 422424, 422435) ('END', 424385, 424390)
('BEGINNING', 405794, 405805) ('END', 410220, 410225)
('BEGINNING', 402545, 402556) ('END', 404106, 404111)
('BEGINNING', 389624, 389635) ('END', 394244, 394249)
('BEGINNING', 248178, 248189) ('END', 249999, 250004)
('BEGINNING', 245311, 245322) ('END', 248171, 248176)
('BEGINNING', 177857, 177868) ('END', 181509, 181514)
('BEGINNING', 159048, 159059) ('END', 161089, 161094)
('BEGINNING', 137917, 137928) ('END', 143818, 143823)
('BEGINNING', 56784, 56795) ('END', 61503, 61508)


Now we will need to annotate the parts by the number of times they appeared, this will prevent that the Markov chain learn that a song can have like 7 REFRAIN.

This will also help in finding errors in the lyrics or structure in the corpus

In [27]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [28]:
def add_cumsum_tags(parts):
    counts = {}
    result = []
    for p in parts:
        counts[p] = counts.get(p, 0)
        if counts[p] != 0 :
            result.append(f"{p}_{counts[p]}")
        else : 
            result.append(f"{p}")
        counts[p] += 1
    return result

def transform_to_num_structure(splitted) :
    #This function will serve only if I decide to aggregate more songs from other rapper because we need to have a sufficient nb of sample
    #or we will have a huge nb of single interaction such as couplet_3 refrain_3 ...
    new_splitted = []
    
    for split in splitted :
        interm = add_cumsum_tags(split.split(" "))
        new_splitted.append(interm)

    count_all = {}
        
    for elem in new_splitted :
        for length in range(len(elem)-1) :
            name = elem[length]+" "+elem[length+1]
            interm = count_all.get(name,0)+1
            count_all[name] = interm
    
    return new_splitted, count_all

In [29]:
new_split, count_all = transform_to_num_structure(splitted)

Maybe I will collect data structure of multiple artists and then aggregate to get a better Markov Chain but if I do this, it will be in the future not now

good_parts = re.sub("END_SECTION ", ""," ".join(all_parts)).split(" ")

In [30]:
count_all

{'BEGINNING REFRAIN': 15,
 'REFRAIN COUPLET': 15,
 'COUPLET REFRAIN_1': 16,
 'REFRAIN_1 COUPLET_1': 13,
 'COUPLET_1 OUTRO': 7,
 'BEGINNING COUPLET': 70,
 'COUPLET REFRAIN': 58,
 'REFRAIN COUPLET_1': 59,
 'COUPLET_1 REFRAIN_1': 59,
 'REFRAIN_1 OUTRO': 20,
 'BEGINNING INTRO': 45,
 'INTRO COUPLET': 39,
 'COUPLET OUTRO': 11,
 'COUPLET PONT': 18,
 'PONT REFRAIN': 8,
 'COUPLET_1 PONT_1': 7,
 'PONT_1 REFRAIN_1': 3,
 'REFRAIN_1 COUPLET_2': 23,
 'COUPLET_2 REFRAIN_2': 16,
 'PONT COUPLET_1': 11,
 'INTRO REFRAIN': 5,
 'BEGINNING OUTRO': 2,
 'COUPLET COUPLET_1': 13,
 'REFRAIN OUTRO': 1,
 'REFRAIN PONT': 4,
 'COUPLET_1 REFRAIN_2': 12,
 'REFRAIN_2 COUPLET_3': 3,
 'COUPLET_3 REFRAIN_3': 2,
 'REFRAIN_2 OUTRO': 12,
 'REFRAIN_1 PONT_1': 3,
 'PONT_1 REFRAIN_2': 2,
 'PONT COUPLET': 2,
 'PONT_1 COUPLET_1': 1,
 'REFRAIN_2 COUPLET_2': 2,
 'COUPLET_1 COUPLET_2': 9,
 'COUPLET_2 PONT': 1,
 'PONT OUTRO': 3,
 'COUPLET_2 OUTRO': 8,
 'PONT_1 COUPLET_2': 3,
 'COUPLET_2 PONT_2': 1,
 'PONT_2 REFRAIN_1': 1,
 'REFRAIN_1

In [31]:
def count_each_interact(splitted) :
    
    for i in range(len(splitted)) :
        splitted[i] = splitted[i]+" END"

    good_parts = (" ".join(splitted)).split(" ")

    dict_glob = {}
    for i in range(len(good_parts)-1) :
        try : 
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] += 1
        except :
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] = 1

    beginn = {}
    intro = {}
    couplet = {}
    pont = {}
    refrain = {}
    outro = {}

    order = ["BEGINNING","INTRO", "COUPLET", "PONT", "REFRAIN", "OUTRO", "END"]
    list_dicti = [beginn, intro, couplet, pont, refrain, outro]
    for part in list_dicti :
        for j in order :
            part[j] = 0    
    
    for i in dict_glob.keys() :
        actual = i.split(" ")[0]
        next = i.split(" ")[1]

        if actual == "BEGINNING" :  
            beginn[next] = dict_glob[i]

        elif actual == "INTRO" :
            intro[next] = dict_glob[i]

        elif actual == "COUPLET" :
            couplet[next] = dict_glob[i]
        elif actual == "PONT" :
            pont[next] = dict_glob[i]
        elif actual == "REFRAIN" :
            refrain[next] = dict_glob[i]
        elif actual == "OUTRO" :
            outro[next] = dict_glob[i]

    return list_dicti

In [32]:
all_count = count_each_interact(splitted)

## Some errors can be found in the differents parts, we will remove what we think is impossible

In [33]:
all_count[0]

{'BEGINNING': 0,
 'INTRO': 45,
 'COUPLET': 70,
 'PONT': 0,
 'REFRAIN': 15,
 'OUTRO': 2,
 'END': 0}

In [34]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-4) :
    nn = len(all_parts)-4 -i
    if all_parts[nn-3][0] == "BEGINNING" :
            if all_parts[nn-2][0] == "OUTRO" :
                if all_parts[nn-1][0] == "END_SECTION" :
                     if all_parts[nn][0] == "END" :
                        part_1_corp = corpus_solo[:all_parts[nn-3][1]]
                        part_2_corp = corpus_solo[all_parts[nn][2]:]
                        corpus_solo = part_1_corp+part_2_corp

We consider that it's not possible to have the end of the music after the beginning

Two of the songs have only detect the beginning because the format that indicates the part are different, we discard them for calculating the probability of change of state

In [35]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [36]:
all_count = count_each_interact(splitted)

In [37]:
all_count[0] #begin

{'BEGINNING': 0,
 'INTRO': 45,
 'COUPLET': 70,
 'PONT': 0,
 'REFRAIN': 15,
 'OUTRO': 0,
 'END': 0}

In [38]:
all_count[1] #intro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 40,
 'PONT': 1,
 'REFRAIN': 5,
 'OUTRO': 0,
 'END': 0}

In [39]:
all_count[2] #couplet

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 24,
 'PONT': 27,
 'REFRAIN': 163,
 'OUTRO': 26,
 'END': 26}

In [40]:
all_count[3] #pont

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 17,
 'PONT': 0,
 'REFRAIN': 16,
 'OUTRO': 4,
 'END': 0}

In [41]:
all_count[4] #refrain

{'BEGINNING': 0,
 'INTRO': 1,
 'COUPLET': 115,
 'PONT': 9,
 'REFRAIN': 5,
 'OUTRO': 33,
 'END': 41}

In [42]:
all_count[4]["REFRAIN"] = 0

Intro isn't an error because I found songs where there was two intro and one for each rapper

In [43]:
all_count[5] #outro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 0,
 'PONT': 0,
 'REFRAIN': 0,
 'OUTRO': 0,
 'END': 63}

## Now we need to obtain the proba for each state to go to another

In [44]:
matrix = []
for i in all_count :
    inter_matrix = []
    for j in i :
        inter_matrix.append(i[j]/sum(i.values()))
    matrix.append(inter_matrix)

In [45]:
order = ["<BEGINNING>","<INTRO>", "<COUPLET>", "<PONT>", "<REFRAIN>", "<OUTRO>", "<END>"]

In [46]:
print(order[:])
print(matrix)

['<BEGINNING>', '<INTRO>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<OUTRO>', '<END>']
[[0.0, 0.34615384615384615, 0.5384615384615384, 0.0, 0.11538461538461539, 0.0, 0.0], [0.0, 0.0, 0.8695652173913043, 0.021739130434782608, 0.10869565217391304, 0.0, 0.0], [0.0, 0.0, 0.09022556390977443, 0.10150375939849623, 0.6127819548872181, 0.09774436090225563, 0.09774436090225563], [0.0, 0.0, 0.4594594594594595, 0.0, 0.43243243243243246, 0.10810810810810811, 0.0], [0.0, 0.005025125628140704, 0.5778894472361809, 0.04522613065326633, 0.0, 0.1658291457286432, 0.20603015075376885], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]


In [47]:
def generate_a_song_structure(matrix) :
    
    song_struct = [0]

    index = [i for i, p in enumerate(matrix[0]) if p>0]
    proba = [p for p in matrix[0] if p > 0]

    cumsum = np.cumsum(proba)
    r = np.random.rand()

    idx = np.searchsorted(cumsum, r)
    selected_value = index[idx] 
    song_struct.append(selected_value)

    end = False

    while end == False :

        index = [i for i, p in enumerate(matrix[selected_value]) if p>0]
        proba = [p for p in matrix[selected_value] if p > 0]

        cumsum = np.cumsum(proba)
        r = np.random.rand()

        idx = np.searchsorted(cumsum, r)
        selected_value = index[idx]
        song_struct.append(selected_value)

        if selected_value == 6 :
            end = True

    print([order[i] for i in song_struct])

In [48]:
for run in range(10) :
    generate_a_song_structure(matrix)

['<BEGINNING>', '<REFRAIN>', '<COUPLET>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<COUPLET>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<END>']
['<BEGINNING>', '<COUPLET>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<OUTRO>', '<END>']
['<BEGINNING>', '<INTRO>', '<REFRAIN>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<COUPLET>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<COUPLET>', '<REFRAIN>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<END>']
['<BEGINNING>', '<INTRO>', '<COUPLET>', '<COUPLET>', '<REFRAIN>', '<OUTRO>', '<END>']


This seems ok for now. 

So the matrix will be exported and used during the model generation of text

In [50]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose()))

,0,1,2,3,4,5,6
0,0.0,0.346154,0.538462,0.0,0.115385,0.0,0.0
1,0.0,0.0,0.869565,0.021739,0.108696,0.0,0.0
2,0.0,0.0,0.090226,0.101504,0.612782,0.097744,0.097744
3,0.0,0.0,0.459459,0.0,0.432432,0.108108,0.0
4,0.0,0.005025,0.577889,0.045226,0.0,0.165829,0.20603
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0,<BEGINNING>,<INTRO>,<COUPLET>,<PONT>,<REFRAIN>,<OUTRO>,<END>


In [51]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose())).to_csv("transition_matrix.csv", index=False)

In [52]:
pd.read_csv("transition_matrix.csv").iloc[5:].values

array([['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '1.0'],
       ['<BEGINNING>', '<INTRO>', '<COUPLET>', '<PONT>', '<REFRAIN>',
        '<OUTRO>', '<END>']], dtype=object)